In [1]:
# Imports
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("/projeto")

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from conf.spark_session import get_spark_session
from delta.tables import DeltaTable

# Sessão do Spark com Delta Lake
spark = get_spark_session()

# Removendo os logs do notebook
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-92899d5c-c372-40c8-818e-44b6213d1283;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 319ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

In [2]:
# Função para visualização dos dados
def show_df(df, n=5):
    display(df.limit(n).toPandas())

In [ ]:
# Caminho dos metadados
metadata_path = "s3a://datalake/bronze/metadata/*/metadata.json"

# Leitura dos metadados
df_metadata = (
    spark.read.
    option("multiLine", True)
    .json(metadata_path)

) 

show_df(df_metadata)

In [ ]:
# Preparação do dataframe de controle
df_execucao = (
    df_metadata
    .select(
        "execution_id",
        "pipeline",
        "source",
        "ingestion_date",
        "execution_timestamp",
        "total_files",
        "status",
        "error_message"
    )
    .withColumn("processed", F.lit(False))
    .withColumn("processed_timestamp", F.lit(None).cast("timestamp"))
    .withColumn("created_at", F.current_timestamp())
)

show_df(df_execucao)

In [ ]:
# Removendo duplicação de dados pelo id de execução
df_execucao_dedup = df_execucao.dropDuplicates(["execution_id"])

show_df(df_execucao_dedup)

In [ ]:
# Caminho da tabela Delta
table_path = "s3a://datalake/silver/controle_execucao"

In [ ]:
# Criar a tabela caso não exista
if not DeltaTable.isDeltaTable(spark, table_path):

    (
        df_execucao_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .save(table_path)
    )

In [ ]:
# Merge incremental
# se execution_id não existir, insere
# se existir, ignora
delta_table = DeltaTable.forPath(spark, table_path)

(
    delta_table.alias("target")
    .merge(
        df_execucao_dedup.alias("source"),
        "target.execution_id = source.execution_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
# Lendo a tabela delta salva na camada silver
df_controle_execucao = spark.read.format("delta").load(table_path)

show_df(df_controle_execucao)

In [ ]:
# Criando o database caso não exista no Hive
spark.sql("""
CREATE DATABASE IF NOT EXISTS lakehouse
""")

In [ ]:
# Verificar os databases no catalogo Hive
spark.sql("SHOW DATABASES").show()

In [ ]:
# Registrar a tabela Delta no Hive Metastore
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS lakehouse.controle_execucao
    USING DELTA
    LOCATION '{table_path}'
""")

In [ ]:
# Validação da tabela
spark.sql("""
    SHOW TABLES IN lakehouse
""").show()

In [ ]:
# Lendo a tabela do Hive com Spark
df = spark.table("lakehouse.controle_execucao")

show_df(df)

FIM